# 🍊 Orange Quant 快速入门

基于 Microsoft qlib 的量化交易算法框架。

本教程将带你完成一个完整的量化实验：
1. 环境检查
2. 数据下载
3. 模型训练
4. 回测评估
5. 自定义策略

## Step 1: 环境检查

确保 qlib 和 orange_quant 能正常导入。

In [6]:
import qlib
import orange_quant

print(f"qlib version: {qlib.__version__ if hasattr(qlib, '__version__') else 'installed'}")
print(f"orange_quant version: {orange_quant.__version__}")

qlib version: 0.9.8.dev31
orange_quant version: 0.1.0


## Step 2: 下载数据

首次运行需要下载中国A股历史日线数据（约 1-2 GB），请耐心等待。

如果已有数据，此步骤会跳过。

In [7]:
from orange_quant.data import download_cn_data

download_cn_data()

[orange_quant] 数据目录已存在: /Users/yuanchengcheng/.qlib/qlib_data/cn_data
[orange_quant] 如需重新下载，请删除该目录后重试。


## Step 3: 运行完整实验

从 YAML 配置加载实验，一键完成：数据加载 → 模型训练 → 回测

In [ ]:
from orange_quant.workflow.experiment import run_from_yaml

# 运行实验
results = run_from_yaml("../config/csi300-lgb-momtopk.yaml")

print(f"\n预测样本数: {len(results['predictions'])}")
print(f"预测样本前5条:")
print(results['predictions'].head())

## Step 4: 查看实验结果

实验自动记录了模型参数、IC 指标、回测绩效等。

In [9]:
recorder = results["recorder"]

# 模型参数
print("模型参数:")
for k, v in recorder.list_params().items():
    if k != "cmd-sys.argv":
        print(f"  {k}: {v}")

# IC 指标
print("\n信号分析指标:")
for k, v in recorder.list_metrics().items():
    if "IC" in k or "Rank" in k:
        print(f"  {k}: {v:.4f}")

# 回测绩效
print("\n回测绩效指标:")
for k, v in recorder.list_metrics().items():
    if "1day" in k:
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# 存储的 artifacts
print("\n实验 Artifacts:")
for a in recorder.list_artifacts():
    print(f"  {a}")

模型参数:
  test_period: 2017-01-01_2020-08-01
  loss: mse
  num_boost_round: 500
  learning_rate: 0.05
  early_stopping_rounds: 50
  instruments: csi300
  bagging_freq: 5
  verbose_eval: 50
  feature_fraction: 0.8
  num_leaves: 64
  bagging_fraction: 0.8
  train_period: 2010-01-01_2014-12-31

信号分析指标:
  Rank IC: 0.0388
  ICIR: 0.1968
  IC: 0.0274
  Rank ICIR: 0.3006

回测绩效指标:
  1day.excess_return_without_cost.std: 0.0063
  1day.pos: 0.0000
  1day.pa: 0.0000
  1day.ffr: 1.0000
  1day.excess_return_without_cost.information_ratio: 0.5651
  1day.excess_return_with_cost.information_ratio: 0.1055
  1day.excess_return_without_cost.max_drawdown: -0.1529
  1day.excess_return_with_cost.max_drawdown: -0.1671
  1day.excess_return_without_cost.mean: 0.0002
  1day.excess_return_with_cost.mean: 0.0000
  1day.excess_return_with_cost.annualized_return: 0.0103
  1day.excess_return_without_cost.annualized_return: 0.0553
  1day.excess_return_with_cost.std: 0.0063

实验 Artifacts:
  code_cached.txt
  code_diff.tx

## Step 5: 自定义策略

继承 `BaseQuantStrategy`，实现自己的交易逻辑。

示例：一个简单的均值回归策略。

In [10]:
from orange_quant.strategies import BaseQuantStrategy
from qlib.backtest.decision import Order, TradeDecisionWO

class MeanReversionStrategy(BaseQuantStrategy):
    """
    均值回归策略示例。
    
    买入前5日跌幅最大的股票，卖出前5日涨幅最大的股票。
    """
    
    def __init__(self, lookback=5, top_n=10, **kwargs):
        super().__init__(**kwargs)
        self.lookback = lookback
        self.top_n = top_n
    
    def generate_trade_decision(self, execute_result=None):
        # TODO: 实现你的交易逻辑
        # 1. 计算过去N日收益率
        # 2. 买入跌最多的，卖出涨最多的
        # 3. 返回 Order 列表
        
        orders = []
        # ... 你的逻辑 ...
        return TradeDecisionWO(orders, self)

print("自定义策略模板已创建！替换其中的 TODO 即可。")

自定义策略模板已创建！替换其中的 TODO 即可。


## 下一步

- 📖 阅读 [qlib 文档](https://qlib.readthedocs.io/)
- 🔧 修改 `config/csi300-lgb-momtopk.yaml` 调整实验参数
- 🧪 在 `orange_quant/strategies/` 下开发自己的策略
- 📊 使用 `orange_quant/backtest/runner.py` 独立运行回测

## 下一步

- 📖 阅读 [qlib 文档](https://qlib.readthedocs.io/)
- 🔧 修改 `config/csi300-lgb-momtopk.yaml` 调整实验参数
- 🧪 在 `orange_quant/strategies/` 下开发自己的策略
- 📊 使用 `orange_quant/backtest/runner.py` 独立运行回测

---

## Step 6: 深度学习时序模型 🆕

orange_quant 支持所有 qlib PyTorch 模型，一行代码切换。
试试 LSTM / GRU / Transformer：</cell id="summary">
<cell id="step6_code">from orange_quant.workflow.experiment import run_dl_from_yaml

# 运行 LSTM 实验（首次运行需要下载预训练权重）
dl_results = run_dl_from_yaml("../config/csi300-lstm-momtopk.yaml")

print(f"\n预测样本数: {len(dl_results['predictions'])}")
print(f"预测前5条:\n{dl_results['predictions'].head()}")

# 信号分析
recorder = dl_results["recorder"]
print("\n信号分析指标:")
for k, v in recorder.list_metrics().items():
    if "IC" in k or "Rank" in k:
        print(f"  {k}: {v:.4f}")</cell id="step6_code">

LSTM 实验运行中（训练较慢，耐心等待）...</cell id="step6_output">
<cell id="step6_note"><cell_type>markdown</cell_type>**支持的所有模型：**
- 时序模型：`LSTM`, `GRU`, `ALSTM`, `Transformer`, `TCN`, `KRNN`
- 注意力模型：`TRA`, `Localformer`, `SFM`
- 图模型：`GATs`, `HIST`, `IGMTF`, `TCTS`
- 其他：`Sandwich`, `ADD`, `ADARNN`

**切换模型只需修改 YAML 中的 `model.name` 和 `model.kwargs`。**
配置文件已备好：`config/csi300-*-momtopk.yaml`</cell id="step6_note">